In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')


In [ ]:
import asyncio
import collections
import concurrent.futures
import json
import re
import time
import litellm
import pandas as pd
import torch
import yaml
from pathlib import Path
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import EvalConfig, build_submission_limits, ALLOWED_ADAPTER_EXTENSIONS
from swegemma.evaluate import Evaluator
from swegemma.models import load_tasks

litellm.drop_params = True

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
DECLARED_MODEL = 'gemma-4-e4b-it-qat-w4a16-ct'
MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{DECLARED_MODEL}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}'

SOUND12_IDS = ["fastapi_14297", "fastapi_14301", "fastapi_14360", "fastapi_14361", "fastapi_14372", "fastapi_14458", "fastapi_14463", "fastapi_14786", "httpx_3672", "requests_6644", "requests_7205", "requests_7309", "requests_7427", "requests_7505", "rich_2725", "rich_2943", "rich_3006", "rich_3130", "rich_3296", "rich_3506", "rich_3518", "rich_3718", "rich_3894", "rich_3905"]
VARIANTS = {
  "v6b_single_clean": {
    "agent.yaml": "name: gemma4_e4b_v6b_single_clean\nmodel: gemma-4-e4b-it-qat-w4a16-ct\ndescription: Single-agent E4B think=0 with 60 tool calls, heredoc edit recovery, and real targeted repo pytest.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\ngenerate_content_config: !include configs/sampling.yaml\n",
    "configs/sampling.yaml": "temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 4096\nthinking_config:\n  thinking_budget: 0\n  include_thoughts: false\n",
    "eval_config.yaml": "evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 60\n  max_time_minutes: 8\n  max_turns: 120\n",
    "prompts/system.md": "You are an autonomous software engineer fixing an issue in `/workspace`.\nYour ONLY tools are: `run_command`, `read_file`, `edit_file`, `write_file`, `get_status`, `submit_patch`. Always run shell commands via `run_command(command=...)`.\nGoal: make the smallest correct edit to SOURCE code in `/workspace` so the hidden test suite passes, verify with real targeted tests, then call `submit_patch`.\n\n## Workflow (Budget: 60 tool calls, 8 minutes)\n1. **Locate source + matching test file fast (calls 1\u20136)**:\n   - Call `run_command` with `git grep -n -F -e '<Symbol1>' -e '<Symbol2>' -- '*.py' | head -25`.\n   - Find the relevant existing test file: `find tests -name '*<module>*' | head -15`.\n   - Call `read_file` on 80\u2013120 lines around the target implementation. For FastAPI doc-code issues, edit executable files under `docs_src/`.\n2. **Edit cleanly (by call 12 \u2014 reliable editing rules)**:\n   - Copy `old_string` **verbatim** from fresh `read_file` or `sed -n 'A,Bp' <file>` output (short 3\u201312 lines, exact indentation, NO manual `\\n`/`\\\"` escapes, `allow_multiple=false`).\n   - **After ONE failed `edit_file`**, immediately switch to a single-quoted Python heredoc via `run_command`:\n     ```bash\n     python3 - <<'EOF'\n     from pathlib import Path\n     p = Path(\"path/to/file.py\")\n     s = p.read_text()\n     old = \"\"\"exact old block\"\"\"\n     new = \"\"\"exact new block\"\"\"\n     assert s.count(old) == 1, f\"count={s.count(old)}\"\n     p.write_text(s.replace(old, new, 1))\n     EOF\n     ```\n   - NEVER use multiline `python3 -c \"...\"` (quotes break). ALWAYS use `python3 - <<'EOF' ... EOF`.\n   - NEVER call `write_file` on `/tmp/...` or to create scratch/test scripts.\n3. **Verify against REAL code & repo tests (no fake mocks!)**:\n   - Run `python3 -m py_compile <edited_file>` AND run the targeted existing test file: `pytest tests/path/to/test_file.py -x -q --tb=short` (use `-k <name>` if huge).\n   - Also test the reported bug + 2 edge cases by importing the real `/workspace` module via `python3 - <<'EOF' ... EOF`.\n   - If `pytest` is disabled in the sandbox, do NOT retry it\u2014rely on the real-import heredoc check. Ignore unrelated pre-existing missing-fixture failures.\n   - If any test or edge case fails, inspect the traceback, refine the source edit, and re-run `py_compile` + the test.\n4. **Submit cleanly**:\n   - Verify `git status -s` shows ONLY intended source edits (remove any stray untracked files, revert any accidental test/config edits).\n   - Call `submit_patch` as your final tool action, confirm `patch_size > 0` and `files_changed > 0`, and write a 1-sentence summary.\n\n## Hard Rules\n- NEVER modify test files (`tests/`, `test_*.py`, `*_test.py`) or configs (`conftest.py`, `pytest.ini`, `pyproject.toml`, `setup.cfg`, `tox.ini`).\n- NEVER run bare `pytest` (`pytest .`), `pip install`, or `git commit`.\n- NEVER finish without calling `submit_patch` on a compiled, non-empty source fix.\n"
  }
}
tasks = load_tasks(TASKS_PATH)
wanted = set(SOUND12_IDS)
subset = sorted([t for t in tasks if t.instance_id in wanted], key=lambda t: t.instance_id)
assert len(subset) == len(SOUND12_IDS), f'Expected {len(SOUND12_IDS)}, got {len(subset)}'

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=False,
    max_loras=1,
    max_lora_rank=128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=(),
)
server_instance = VllmServer(vllm_cfg)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})', flush=True)

models = server_instance.create_model_registry(
    aliases=[DECLARED_MODEL],
    model_prefix='openai/',
    api_key=os.environ.get('LOCAL_API_KEY', 'EMPTY'),
)

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if (callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn)) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

limits, gen_constraints = build_submission_limits()
all_summaries = []

for run_name, files_map in VARIANTS.items():
    print(f'\n================ RUNNING {run_name} ({len(subset)} tasks) ================', flush=True)
    run_dir = WORKING_DIR / run_name
    agent_dir = run_dir / 'agent'
    if agent_dir.exists():
        shutil.rmtree(agent_dir)
    for rel, content in files_map.items():
        p = agent_dir / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content, encoding='utf-8')

    ev_raw = yaml.safe_load((agent_dir / 'eval_config.yaml').read_text(encoding='utf-8')) or {}
    ev = ev_raw.get('evaluation', ev_raw)
    max_time_minutes = float(ev.get('max_time_minutes', 8.0))
    adapters = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

    eval_config = EvalConfig(
        tasks_path=TASKS_PATH,
        snapshots_dir=DATA_DIR / 'snapshots',
        results_dir=run_dir / 'results',
        submission_dir=agent_dir,
        models=models,
        sandbox='subprocess',
        timeout_seconds=int(ev.get('timeout_seconds', 240)),
        verification_timeout_seconds=int(ev.get('verification_timeout_seconds', 300)),
        max_time_minutes=max_time_minutes,
        max_tool_calls=int(ev.get('max_tool_calls', 28)),
        max_turns=int(ev.get('max_turns', 80)),
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
        graph_dir=GRAPH_DIR,
        embeddings_dir=EMBEDDINGS_DIR,
        wheels_dir=DATA_DIR / 'wheels',
        verbose=False,
    )
    evaluator = Evaluator(eval_config)
    rows_path = run_dir / 'per_task.jsonl'
    rows = []
    for idx, task in enumerate(subset, start=1):
        started = time.perf_counter()
        try:
            result = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(subset))
            patch = (result.agent_patch or '').strip()
            row = dict(
                run=run_name, id=task.instance_id, repo=task.repo,
                resolved=bool(result.resolved), patch_chars=len(patch),
                tool_calls=result.tool_calls,
                agent_seconds=round(float(result.duration_seconds or 0.0), 1),
                test_exit_code=result.test_exit_code,
                error=getattr(result, 'error', None),
            )
        except Exception as exc:
            row = dict(
                run=run_name, id=task.instance_id, repo=task.repo,
                resolved=False, patch_chars=0, tool_calls=None,
                agent_seconds=None, test_exit_code=None, error=repr(exc)[:300],
            )
        row['wall_seconds'] = round(time.perf_counter() - started, 1)
        rows.append(row)
        with rows_path.open('a') as f:
            f.write(json.dumps(row) + '\n')
        print(f"[{run_name} {idx}/{len(subset)}] {row['id']}: resolved={row['resolved']} "
              f"patch={row['patch_chars']} calls={row['tool_calls']} "
              f"agent={row['agent_seconds']}s wall={row['wall_seconds']}s err={row['error']}", flush=True)

    df = pd.DataFrame(rows)
    agent_s = df['agent_seconds'].dropna()
    summary = {
        'run': run_name,
        'tasks': len(df),
        'resolved': int(df['resolved'].sum()),
        'resolution_rate': round(float(df['resolved'].mean()), 3),
        'no_patch': int((df['patch_chars'] == 0).sum()),
        'errors': int(df['error'].notna().sum()),
        'mean_tool_calls': round(float(df['tool_calls'].dropna().mean()), 1) if len(df['tool_calls'].dropna()) else None,
        'agent_seconds_median': float(agent_s.median()) if len(agent_s) else None,
        'wall_seconds_mean': round(float(df['wall_seconds'].mean()), 1),
    }
    (run_dir / 'summary.json').write_text(json.dumps(summary, indent=2))
    all_summaries.append(summary)
    print(json.dumps(summary, indent=2), flush=True)
    print(df.to_string(index=False), flush=True)

(WORKING_DIR / 'ablation_summary.json').write_text(json.dumps(all_summaries, indent=2))
print('\n=== FINAL ABLATION COMPARISON ===', flush=True)
print(pd.DataFrame(all_summaries).to_string(index=False), flush=True)
